## Metadata analysis of sebum and ph levels collected from participants


Date created: 10/23/2024

Notebook author: Britta De Pessemier 

Data analysis by:  Britta De Pessemier


In [67]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import mannwhitneyu
import biom
from biom import load_table
import seaborn as sns
from scipy.stats import spearmanr
import numpy as np

In [68]:
# Read the metadata file
metadata_file = '../Metadata/metadata_final_22102024.tsv'
metadata_df = pd.read_csv(metadata_file, sep='\t')

In [69]:
# Define the color palette for the severity groups
severity_group_colors = {
    "absent Healthy": "#000096",       # Dark Blue for Healthy
    "absent Acne_NL": "#17c6d5",       # Light Blue for Acne Non-lesional (absent)
    "low Acne_L": "#ff676c",           # Light Red for Low Acne Lesional
    "moderate Acne_L": "#ca0000",      # Darker Red for Moderate Acne Lesional
    "high Acne_L": "#960000"           # Darkest Red for High Acne Lesional
}

# Ensure the severity groups are ordered as requested
metadata_df['severity_group'] = pd.Categorical(metadata_df['severity_group'], 
                                                categories=['absent Healthy', 'absent Acne_NL', 'low Acne_L', 'moderate Acne_L', 'high Acne_L'], 
                                                ordered=True)

# Create two subplots next to each other but without sharing the y-axis
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 8))

# First subplot for skin pH (ax1)
sns.boxplot(x='severity_group', y='skin_ph_meter_ph_mean_forehead_right', data=metadata_df, palette=severity_group_colors, ax=ax1)
sns.stripplot(x='severity_group', y='skin_ph_meter_ph_mean_forehead_right', data=metadata_df, palette=severity_group_colors, jitter=True, dodge=False, ax=ax1, linewidth=0.6)
ax1.set_ylabel('Skin pH (Forehead Right)', fontsize=16)
ax1.set_xlabel('')  # No x-axis label
ax1.set_title('Skin pH by Severity Group', fontsize=18)

# Second subplot for sebum level (ax2)
sns.boxplot(x='severity_group', y='sebumeter_casual_level_mean_forehead_left', data=metadata_df, palette=severity_group_colors, ax=ax2)
sns.stripplot(x='severity_group', y='sebumeter_casual_level_mean_forehead_left', data=metadata_df, palette=severity_group_colors, jitter=True, dodge=False, ax=ax2, linewidth=0.6)
ax2.set_ylabel('Sebum Level (Forehead Left)', fontsize=16)
ax2.set_xlabel('')  # No x-axis label
ax2.set_title('Sebum Level by Severity Group', fontsize=18)

# Custom x-axis labels to avoid overlap
new_labels = ['Healthy', 'Acne\nNon-lesional', 'Low Acne\nLesional', 'Moderate Acne\nLesional', 'High Acne\nLesional']
ax1.set_xticklabels(new_labels, rotation=20, ha='right', fontsize=12)
ax2.set_xticklabels(new_labels, rotation=20, ha='right', fontsize=12)

# Function to draw significance brackets closer to the boxplots and one another
def add_significance(ax, x1, x2, y_start, p_value, height_step, significance_offset):
    # Set significance label
    if p_value >= 0.05:
        label = 'ns'
    elif p_value < 0.001:
        label = '***'
    elif p_value < 0.01:
        label = '**'
    else:
        label = '*'

    # Draw the significance brackets
    y = y_start + height_step  # Vertical position for the horizontal line
    ax.plot([x1, x1, x2, x2], [y, y + significance_offset, y + significance_offset, y], lw=1, color='black')
    ax.text((x1 + x2) * 0.5, y + significance_offset * 0.75, label, ha='center', va='bottom', fontsize=12)

# Pairwise significance testing for the skin pH plot (ax1)
severity_groups = ['absent Healthy', 'absent Acne_NL', 'low Acne_L', 'moderate Acne_L', 'high Acne_L']
p_values_ph = {}

# Heights to draw the annotation lines for skin pH
y_max_ph = max(metadata_df['skin_ph_meter_ph_mean_forehead_right']) + 0.05  # Reduce initial height for closer brackets
height_step_ph = 0.05  # Reduce the height between brackets
significance_offset = 0.01  # Bring the significance label closer to the bracket

# Perform pairwise comparisons and plot only significant results for skin pH
for i, group1 in enumerate(severity_groups):
    for j, group2 in enumerate(severity_groups):
        if i < j:
            # Get the skin pH values for each severity group
            group1_values_ph = metadata_df[metadata_df['severity_group'] == group1]['skin_ph_meter_ph_mean_forehead_right']
            group2_values_ph = metadata_df[metadata_df['severity_group'] == group2]['skin_ph_meter_ph_mean_forehead_right']
            
            # Perform Mann-Whitney U test
            stat, p_ph = mannwhitneyu(group1_values_ph, group2_values_ph, alternative='two-sided')
            p_values_ph[f'{group1} vs {group2}'] = p_ph
            
            # Plot only if p-value is significant (p < 0.05)
            if p_ph < 0.05:
                add_significance(ax1, i, j, y_max_ph, p_ph, height_step_ph, significance_offset)
                y_max_ph += height_step_ph  # Increment height after each bracket

# Pairwise significance testing for the sebum level plot (ax2)
p_values_sebum = {}

# Heights to draw the annotation lines for sebum level
y_max_sebum = max(metadata_df['sebumeter_casual_level_mean_forehead_left']) + 0.05  # Reduce initial height
height_step_sebum = 10  # Adjust the height between brackets based on the sebum data's larger range

# Perform pairwise comparisons and plot only significant results for sebum level
for i, group1 in enumerate(severity_groups):
    for j, group2 in enumerate(severity_groups):
        if i < j:
            # Get the sebum level values for each severity group
            group1_values_sebum = metadata_df[metadata_df['severity_group'] == group1]['sebumeter_casual_level_mean_forehead_left']
            group2_values_sebum = metadata_df[metadata_df['severity_group'] == group2]['sebumeter_casual_level_mean_forehead_left']
            
            # Perform Mann-Whitney U test
            stat, p_sebum = mannwhitneyu(group1_values_sebum, group2_values_sebum, alternative='two-sided')
            p_values_sebum[f'{group1} vs {group2}'] = p_sebum
            
            # Plot only if p-value is significant (p < 0.05)
            if p_sebum < 0.05:
                add_significance(ax2, i, j, y_max_sebum, p_sebum, height_step_sebum, significance_offset)
                y_max_sebum += height_step_sebum  # Increment height after each bracket

# Set different y-limits for the two subplots to avoid overlap of significance brackets
ax1.set_ylim(4.35, 6.1)  # Skin pH range
ax2.set_ylim(0, y_max_sebum + height_step_sebum)  # Sebum level range

# Save the figure
# plt.savefig(f'../Figures/clinical_metadata_Figures/skin_ph_sebum_forehead_side_by_side_fixed_yaxis.png', dpi=600)  # Save as png
# plt.savefig(f'../Figures/clinical_metadata_Figures/skin_ph_sebum_forehead_side_by_side_fixed_yaxis.svg')  # Save as svg

# Print pairwise p-values in scientific notation for skin pH
# print("Pairwise Mann-Whitney U test p-values (skin pH):")
# for comparison, p_value in p_values_ph.items():
#     print(f"{comparison}: p-value = {p_value:.2e}")

# # Print pairwise p-values in scientific notation for sebum level
# print("Pairwise Mann-Whitney U test p-values (sebum level):")
# for comparison, p_value in p_values_sebum.items():
#     print(f"{comparison}: p-value = {p_value:.2e}")


/Users/yangchen/miniforge3/envs/qiime2-metagenome-2024.10/lib/python3.10/site-packages/seaborn/categorical.py:641: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  grouped_vals = vals.groupby(grouper)
/var/folders/22/yck9vwx53w1c38tvj_c0_tz00000gn/T/ipykernel_40452/1832883060.py:20: FutureWarning: Passing `palette` without assigning `hue` is deprecated.
  sns.stripplot(x='severity_group', y='skin_ph_meter_ph_mean_forehead_right', data=metadata_df, palette=severity_group_colors, jitter=True, dodge=False, ax=ax1, linewidth=0.6)
/Users/yangchen/miniforge3/envs/qiime2-metagenome-2024.10/lib/python3.10/site-packages/seaborn/_oldcore.py:1119: FutureWarning: use_inf_as_na option is deprecated and will be removed in a future version. Convert inf values to NaN before operating instead.
  with pd.option_contex

(0.0, 305.05)

# healthy vs acne

In [70]:
# Combine Acne_NL and Acne_L groups into a single 'Acne' group in the metadata
metadata_df['severity_group_combined'] = metadata_df['severity_group'].replace({
    'absent Acne_NL': 'Acne',
    'low Acne_L': 'Acne',
    'moderate Acne_L': 'Acne',
    'high Acne_L': 'Acne',
    'absent Healthy': 'Healthy'
})

# Define the color palette for the two groups: Healthy and Acne
severity_group_colors_combined = {
    "Healthy": "#423fa6",  # Dark Blue for Healthy
    "Acne": "#b11919"      # Red for combined Acne group
}

# Ensure the groups are ordered properly
metadata_df['severity_group_combined'] = pd.Categorical(metadata_df['severity_group_combined'], 
                                                        categories=['Healthy', 'Acne'], 
                                                        ordered=True)

# Create two subplots next to each other but without sharing the y-axis
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 10))  # ax1 (left), ax2 (right)

# First subplot for sebum level (ax1)
sns.boxplot(x='severity_group_combined', y='sebumeter_casual_level_mean_forehead_left', 
            data=metadata_df, palette=severity_group_colors_combined, ax=ax1)
sns.stripplot(x='severity_group_combined', y='sebumeter_casual_level_mean_forehead_left', 
              data=metadata_df, palette=severity_group_colors_combined, jitter=True, dodge=False, ax=ax1, linewidth=0.6)
ax1.set_ylabel('Sebum Level (Forehead Left)', fontsize=16)
ax1.set_xlabel('')  # No x-axis label
ax1.set_title('Sebum Level by Cohort', fontsize=18)

# Second subplot for skin pH (ax2)
sns.boxplot(x='severity_group_combined', y='skin_ph_meter_ph_mean_forehead_right', 
            data=metadata_df, palette=severity_group_colors_combined, ax=ax2)
sns.stripplot(x='severity_group_combined', y='skin_ph_meter_ph_mean_forehead_right', 
              data=metadata_df, palette=severity_group_colors_combined, jitter=True, dodge=False, ax=ax2, linewidth=0.6)
ax2.set_ylabel('Skin pH (Forehead Right)', fontsize=16)
ax2.set_xlabel('')  # No x-axis label
ax2.set_title('Skin pH by Cohort', fontsize=18)

# Set the x-axis labels
ax1.set_xticklabels(['Healthy', 'Acne'], rotation=0, ha='center', fontsize=14)
ax2.set_xticklabels(['Healthy', 'Acne'], rotation=0, ha='center', fontsize=14)

# Function to add only the significance label without horizontal or vertical lines
def add_significance_label(ax, x1, x2, p_value, y):
    # Determine the significance label based on p-value
    if p_value >= 0.05:
        label = f"ns  {p_value:.2e}"
    elif p_value < 0.001:
        label = f"***  {p_value:.2e}"
    elif p_value < 0.01:
        label = f"**  {p_value:.2e}"
    else:
        label = f"*  {p_value:.2e}"
    
    # Annotate the significance label above the two boxplots, centered between x1 and x2
    ax.text((x1 + x2) * 0.5, y, label, ha='center', va='bottom', fontsize=12)

# Adjust y-axis limits and add significance labels
y_sebum = max(metadata_df['sebumeter_casual_level_mean_forehead_left']) + 8
y_ph = max(metadata_df['skin_ph_meter_ph_mean_forehead_right']) + 0.05

# Add only the labels without any lines
add_significance_label(ax1, x1=0, x2=1, p_value=p_sebum, y=y_sebum)
add_significance_label(ax2, x1=0, x2=1, p_value=p_ph, y=y_ph)

# Set y-limits for better readability
ax1.set_ylim(0, max(metadata_df['sebumeter_casual_level_mean_forehead_left']) + 20)  # Adjust for sebum level
ax2.set_ylim(4.35, 6.1)  # Adjust for skin pH

# Save the figure
plt.savefig(f'../Figures/clinical_metadata_Figures/skin_ph_sebum_Healthy_vs_Acne.png', dpi=600)
# plt.savefig(f'../Figures/clinical_metadata_Figures/skin_ph_sebum_Healthy_vs_Acne.svg')

# Store the p-values in a dictionary
p_values_combined = {
    'Sebum Level': p_sebum,
    'Skin pH': p_ph
}

# Print Mann-Whitney U test p-values
print("Pairwise Mann-Whitney U test p-values:")
for comparison, p_value in p_values_combined.items():
    print(f"{comparison}: p-value = {p_value:.2e}")

# Show the plot
plt.tight_layout()
plt.show()


/var/folders/22/yck9vwx53w1c38tvj_c0_tz00000gn/T/ipykernel_40452/2660379322.py:2: FutureWarning: The behavior of Series.replace (and DataFrame.replace) with CategoricalDtype is deprecated. In a future version, replace will only be used for cases that preserve the categories. To change the categories, use ser.cat.rename_categories instead.
  metadata_df['severity_group_combined'] = metadata_df['severity_group'].replace({
/Users/yangchen/miniforge3/envs/qiime2-metagenome-2024.10/lib/python3.10/site-packages/seaborn/categorical.py:641: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  grouped_vals = vals.groupby(grouper)
/var/folders/22/yck9vwx53w1c38tvj_c0_tz00000gn/T/ipykernel_40452/2660379322.py:27: FutureWarning: Passing `palette` without assigning `hue` is deprecated.
  sns.stripplot(x='severity_gro

Pairwise Mann-Whitney U test p-values:
Sebum Level: p-value = 2.33e-01
Skin pH: p-value = 5.59e-01


/var/folders/22/yck9vwx53w1c38tvj_c0_tz00000gn/T/ipykernel_40452/2660379322.py:90: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
